In [6]:
# ============================================================
# BUILD DIM_CONSTRUCTORS — LOCAL GOLD
# ============================================================

import os
from pyspark.sql import functions as F

In [7]:
try:
    SILVER_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:36 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:36 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:36 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [8]:
silver_constructors = f"{SILVER_ROOT}/constructors/constructors_silver.parquet"
ref_nat_region = f"{GOLD_ROOT}/ref_nationality_region/ref_nationality_region.parquet"
gold_folder = f"{GOLD_ROOT}/dim_constructors"
gold_file = f"{gold_folder}/dim_constructors.parquet"
os.makedirs(gold_folder, exist_ok=True)

In [9]:
constructors_df = spark.read.parquet(silver_constructors)
ref_df = spark.read.parquet(ref_nat_region)

In [10]:
dim_constructors_df = (
    constructors_df.alias("c")
        .join(
            ref_df.alias("r"),
            F.col("c.nationality") == F.col("r.nationality"),
            "left"
        )
        .select(
            F.col("c.constructor_id"),
            F.col("c.constructor_name"),
            F.col("c.nationality"),
            F.col("r.region").alias("nationality_region")
        )
)

In [11]:
(
    dim_constructors_df
        .write
        .mode("overwrite")
        .parquet(gold_file)
)

print(f"✔ dim_constructors written to: {gold_file}")
spark.read.parquet(gold_file).show(10, truncate=False)

✔ dim_constructors written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/dim_constructors/dim_constructors.parquet
+--------------+----------------+-------------+------------------+
|constructor_id|constructor_name|nationality  |nationality_region|
+--------------+----------------+-------------+------------------+
|adams         |Adams           |American     |North America     |
|afm           |AFM             |German       |Europe            |
|ags           |AGS             |French       |Europe            |
|alfa          |Alfa Romeo      |Swiss        |Europe            |
|alphatauri    |AlphaTauri      |Italian      |Europe            |
|alpine        |Alpine F1 Team  |French       |Europe            |
|alta          |Alta            |British      |Europe            |
|amon          |Amon            |New Zealander|Oceania           |
|apollon       |Apollon         |Swiss        |Europe            |
|arrows        |Arrows          |British      |Europe            |
+-----------